In [ ]:
import os
import gradio as gr
from dotenv import load_dotenv
from agents import Runner
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, WebSearchTool, ModelSettings, trace
from pydantic import BaseModel, Field
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict

: 

In [9]:
load_dotenv(override=True)

True

In [ ]:
""" Planning a holiday planner """

from agents import function_tool
import sendgrid


CLARIFICATIONS = 5
SEARCH_QUERY = 5

clarification_planner_instructions = f"You are a professional holiday planner. You take into account user's requirements and plan a perfect holiday. \
    You don't want to act hastily and should analyze the plan. So before moving to planning all the requirements from the user needs to be gathered. \
    you need to come up with {CLARIFICATIONS} clarification question to take into account to plan a perfect and memorable holiday for user."

clarification_planner_agent = Agent(
    name="planner_clarification_agent",
    instructions=clarification_planner_instructions,
    model="gpt-4o-mini"
)

search_agent_instructions = "You are a research assistant. Given a search term, you search the web for that term and  \
    produce a concise summary of the results. The summary must 2-3 paragraphs and less than 500  \
    words. Capture the main points. Write succintly, no need to have complete sentences or good  \
    grammar. This will be consumed by someone synthesizing a report, so its vital you capture the  \
    essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=search_agent_instructions,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4.1-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

search_agent_tool = search_agent.as_tool(tool_name="search_agent", tool_description="You are a search agent")

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

holiday_planner_instructions = f"You are a holiday planner who have over a decade of experience in this field. You are provided with user's requirement. \
    You will also be provided with additional clarification questions which were asked to user and answers provided to it. \
    You need to come up with {SEARCH_QUERY} set of web searches to plan the perfect holiday for the user."

holiday_planner_agent = Agent(
    name="holiday_planner",
    instructions=holiday_planner_instructions,
    model="gpt-5-nano",
    output_type=WebSearchPlan
)

holiday_planner_agent_tool = holiday_planner_agent.as_tool(tool_name="holiday_planner_agent", tool_description="You are a holiday planner agent")

class HolidayPlan(BaseModel):
    markdown_report: str = Field(description="The final report")

report_tool_instructions = "You are a senior holiday planner tasked with formating the holiday plan such that it will be easily readable and understandable to the user \
    You will be provided with original query and the final holiday plan. Keep the report concise and easily understandable for the customer."

writer_agent = Agent(
    name="write_agent",
    instructions=report_tool_instructions,
    model="gpt-4o-mini",
    output_type=HolidayPlan
)

writer_agent_tool = writer_agent.as_tool(tool_name="writer_agent", tool_description="You are a report writer")

subject_instructions = "You can write a subject for an email. \
You are given a holiday plan and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a holiday plan email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("neneakshay@gmail.com")  # Change to your verified sender
    to_email = To("neneakshay@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=[subject_tool, html_tool, send_html_email],
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

planner_orchestrator_instructions = f"You are seasoned holiday planner. You will be provided with user's requirements. \
    User your holiday_planner_agent tool to come up with searches you require to plan the holiday \
    Use search_agent tool to search those queries and gather the answers. \
    If you are not satisfied with the information you've got so far to plan the trip, use holiday_planner_agent again to come up with new search queries \
    and use search_agent tool to get results \
    Once all the information is gathered, Create a detailed holiday plan for user. Make sure it should satisfy all the requirements from the user \
    Once plan is finalized provide all the user requirements and \
    final holiday plan to write_agent tool to format the final Holiday Plan which can be handed over to customer. \
    After finalising holiday plan, Use emailer_agent to send this to customer."

trip_planner_agent = Agent(
    name="trip_planner",
    instructions=planner_orchestrator_instructions,
    model="gpt-4.1-mini",
    tools=[holiday_planner_agent_tool, search_agent_tool, writer_agent_tool],
    handoffs=[emailer_agent]
)

In [15]:
async def call_questions_stream(query: str):
    """ Generating clarification questions """
    yield gr.update(visible=True, value="")
    with trace("Clarifying Questions"):
        full_text = ""
        result = Runner.run_streamed(clarification_planner_agent, input=query)
        async for event in result.stream_events():
            if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
                full_text += event.data.delta
                yield full_text

async def generate_holiday_plan(query: str, questions: str, answers: str):
    """ Generating Holida Plan """
    input_query = f"Query: {query}, Clarification questions asked to customer: {questions}, answers to those questions by customer: {answers}"
    
    final_plan = await Runner.run(trip_planner_agent, input=input_query)
    yield gr.update(visible=True, value=final_plan.final_output)


In [16]:
with gr.Blocks() as trip_planner:

    gr.Markdown("Trip Planner Agent")
    gr.Markdown("Please enter your query below and I will try to be helpful and plan a perfect holiday for you")

    with gr.Row():
        with gr.Column(scale=1):

            holiday_query = gr.Textbox(
                label="Where would you like to travel?",
                lines=2)

            submit_button = gr.Button("Start Planning!", variant="primary", size="lg")

            questions = gr.TextArea(visible=False, label="Clarifying Questions")
            answers = gr.Textbox(
                visible=False,
                label="Your Answers",
                placeholder="Please provide answers to above questions to plan your perfect holiday!",
                lines=5)

            generate_plan = gr.Button("Generate Plan", visible=False, variant="primary", size="lg")
        
        with gr.Column(scale=1):
            report = gr.Markdown(label="Final Holiday Plan")

    submit_button.click(
        fn=call_questions_stream,
        inputs=holiday_query,
        outputs=[questions]
    ).then(
        fn=lambda: (gr.update(visible=True, interactive=True), gr.update(visible=True)),
        inputs=None,
        outputs=[answers, generate_plan]
    )
    
    generate_plan.click(
        fn=generate_holiday_plan,
        inputs=[holiday_query, questions, answers],
        outputs=[report]
    )


trip_planner.launch(inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
